The dataset used : https://www.kaggle.com/datasets/paultimothymooney/chest-xray-pneumonia/data

All images are in jpeg format. As we can't use libraries to deal with this format, we have to use pgm format, which can be turned into numpy ndarray for scikit-learn.

Search for : *Application of the AI System for Pneumonia Detection Using Chest X-Ray Images* in http://www.cell.com/cell/fulltext/S0092-8674(18)30154-5

In [41]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import ListedColormap

from sklearn.pipeline import Pipeline
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis
from sklearn.ensemble import AdaBoostClassifier, RandomForestClassifier
from sklearn.gaussian_process import GaussianProcessClassifier
from sklearn.gaussian_process.kernels import RBF
from sklearn.inspection import DecisionBoundaryDisplay
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.dummy import DummyClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report
from skimage.io import imread
from skimage.color import rgb2gray
from skimage.transform import resize

As Scikit-learn only deals with mathematical objects (in this case numpy ndarray), we need to do some operations on our dataset.
Our images are in .jpeg format. We use scikit-image library to convert them into numpy ndarray.

Simple bash script using ImageMagik to resize images. The reason we do it in bash is bash is much faster than python to do this kind of operation. 

find archive/chest_xray/train -type f -name "*.jpeg" -exec mogrify -strip -resize 64x64! {} +



(consider changing resolution)

In [42]:
input_dir = 'archive/chest_xray/train'
categories = ['NORMAL', 'PNEUMONIA']

data = [] # images as numerical data
labels = [] # categories corresponding to each images

for category_idx, category in enumerate(categories):
    folder = os.path.join(input_dir, category)
    for file in os.scandir(folder):
        img_path = file.path
        img = imread(img_path)
        if img.ndim == 3:
            img = rgb2gray(img) # conversion to shade of gray for optimisation
        # img = resize(img, (64,64), anti_aliasing=True).astype(np.float32)
        data.append(img)
        labels.append(category_idx)

data = np.array(data, dtype=np.float32)
data = data.reshape(len(data), -1)
labels = np.array(labels)

# We save the numpy array as .npy so we can use our dataset without restarting this block of code
np.save("data.npy", data) 
np.save("labels.npy", labels)

In [43]:
data = np.load("data.npy")
labels = np.load("labels.npy")

x_train, x_test, y_train, y_test = train_test_split(data, labels, test_size=0.2, shuffle=True, stratify=labels)

consider adding a pipeline

In [44]:
pipeline = Pipeline([
    ('scalify', StandardScaler()),
    ('model', DummyClassifier())
])

In [45]:
def create_grid_search(model_name, classifier, parameters):
    
    pipeline.set_params(model=classifier)

    grid_search = GridSearchCV(
        pipeline,
        param_grid=parameters,
        cv=4,
        n_jobs=-1,
        scoring='accuracy',
        verbose=2
    )
    grid_search.fit(x_train, y_train)

    best_estimator = grid_search.best_estimator_
    y_prediction = best_estimator.predict(x_test)

    score = accuracy_score(y_prediction, y_test)

    print(f"Metrics for {model_name}:")
    print(f"Best parameters: {grid_search.best_params_}")
    print(classification_report(y_test, y_prediction))

    print('{}% of samples were correctly classified'.format(str(score * 100)))

Results tab will store the scores of the different metrics for each model.

In [46]:
results = []

KNeighborsClassifier

Metrics for KNeighborsClassifier:
Best parameters: {'model__n_neighbors': 10}
              precision    recall  f1-score   support

           0       0.93      0.81      0.87       315
           1       0.93      0.98      0.95       854

    accuracy                           0.93      1169
   macro avg       0.93      0.89      0.91      1169
weighted avg       0.93      0.93      0.93      1169



In [47]:
KNeighborsClassifier_parameters = [{'model__n_neighbors' : [3, 5, 10, 15, 20]}]
create_grid_search("KNeighborsClassifier", KNeighborsClassifier(), KNeighborsClassifier_parameters)

Fitting 4 folds for each of 5 candidates, totalling 20 fits
[CV] END ..............................model__n_neighbors=10; total time= 1.3min
[CV] END ...............................model__n_neighbors=3; total time= 1.3min
[CV] END ..............................model__n_neighbors=10; total time= 1.3min
[CV] END ...............................model__n_neighbors=3; total time= 1.3min
[CV] END ...............................model__n_neighbors=3; total time= 1.3min
[CV] END ...............................model__n_neighbors=5; total time= 1.3min
[CV] END ...............................model__n_neighbors=5; total time= 1.3min
[CV] END ...............................model__n_neighbors=3; total time= 1.3min
[CV] END ..............................model__n_neighbors=10; total time= 1.3min
[CV] END ...............................model__n_neighbors=5; total time= 1.3min
[CV] END ..............................model__n_neighbors=10; total time= 1.3min
[CV] END ...............................model__n_

DecisionTreeClassifier


Metrics for DecisionTreeClassifier:
Best parameters: {'model__max_depth': 5, 'model__min_samples_leaf': 1, 'model__min_samples_split': 5}
              precision    recall  f1-score   support

           0       0.81      0.79      0.80       315
           1       0.92      0.93      0.93       854

    accuracy                           0.89      1169
   macro avg       0.87      0.86      0.86      1169
weighted avg       0.89      0.89      0.89      1169

In [48]:
DecisionTreeClassifier_parameters = [{
    "model__max_depth": [5, 10, 20, None],
    "model__min_samples_split": [2, 5, 10],
    "model__min_samples_leaf": [1, 2, 4]
}]
create_grid_search("DecisionTreeClassifier", DecisionTreeClassifier(), DecisionTreeClassifier_parameters)

Fitting 4 folds for each of 36 candidates, totalling 144 fits
[CV] END model__max_depth=5, model__min_samples_leaf=1, model__min_samples_split=10; total time=   9.3s
[CV] END model__max_depth=5, model__min_samples_leaf=1, model__min_samples_split=10; total time=   9.6s
[CV] END model__max_depth=5, model__min_samples_leaf=1, model__min_samples_split=2; total time=   9.6s
[CV] END model__max_depth=5, model__min_samples_leaf=1, model__min_samples_split=5; total time=   9.6s
[CV] END model__max_depth=5, model__min_samples_leaf=1, model__min_samples_split=5; total time=   9.6s
[CV] END model__max_depth=5, model__min_samples_leaf=1, model__min_samples_split=5; total time=   9.7s
[CV] END model__max_depth=5, model__min_samples_leaf=1, model__min_samples_split=2; total time=   9.8s
[CV] END model__max_depth=5, model__min_samples_leaf=1, model__min_samples_split=2; total time=   9.9s
[CV] END model__max_depth=5, model__min_samples_leaf=1, model__min_samples_split=5; total time=  10.3s
[CV] END 

RandomForestClassifier

Metrics for RandomForestClassifier:
Best parameters: {'model__max_depth': 30, 'model__min_samples_leaf': 2, 'model__min_samples_split': 5, 'model__n_estimators': 300}
              precision    recall  f1-score   support

           0       0.94      0.87      0.90       315
           1       0.95      0.98      0.97       854

    accuracy                           0.95      1169
   macro avg       0.95      0.92      0.93      1169
weighted avg       0.95      0.95      0.95      1169

94.9529512403764% of samples were correctly classified

In [49]:
RandomForestClassifier_parameters = [{
    'model__max_depth': [10, 30, None],
    'model__min_samples_leaf': [1, 2, 4],
    'model__min_samples_split': [2, 5, 10],
    'model__n_estimators': [100, 300]
}]
create_grid_search("RandomForestClassifier", RandomForestClassifier(), RandomForestClassifier_parameters)

Fitting 4 folds for each of 54 candidates, totalling 216 fits
[CV] END model__max_depth=10, model__min_samples_leaf=1, model__min_samples_split=2, model__n_estimators=100; total time=  18.4s
[CV] END model__max_depth=10, model__min_samples_leaf=1, model__min_samples_split=5, model__n_estimators=100; total time=  18.9s
[CV] END model__max_depth=10, model__min_samples_leaf=1, model__min_samples_split=5, model__n_estimators=100; total time=  19.0s
[CV] END model__max_depth=10, model__min_samples_leaf=1, model__min_samples_split=2, model__n_estimators=100; total time=  19.1s
[CV] END model__max_depth=10, model__min_samples_leaf=1, model__min_samples_split=5, model__n_estimators=100; total time=  19.4s
[CV] END model__max_depth=10, model__min_samples_leaf=1, model__min_samples_split=5, model__n_estimators=100; total time=  19.5s
[CV] END model__max_depth=10, model__min_samples_leaf=1, model__min_samples_split=2, model__n_estimators=100; total time=  20.3s
[CV] END model__max_depth=10, model

SVC

Metrics for SVC:
Best parameters: {'model__C': 100, 'model__gamma': 0.0001}
              precision    recall  f1-score   support

           0       0.93      0.93      0.93       315
           1       0.98      0.97      0.97       854

    accuracy                           0.96      1169
   macro avg       0.95      0.95      0.95      1169
weighted avg       0.96      0.96      0.96      1169

96.32164242942686% of samples were correctly classified

In [50]:
svc_parameters = [{'model__gamma': [0.01, 0.001, 0.0001], 'model__C': [1, 10, 100, 1000]}]
create_grid_search("SVC", SVC(), svc_parameters)

Fitting 4 folds for each of 12 candidates, totalling 48 fits
[CV] END ....................model__C=1, model__gamma=0.0001; total time= 1.3min
[CV] END ....................model__C=1, model__gamma=0.0001; total time= 1.3min
[CV] END ....................model__C=1, model__gamma=0.0001; total time= 1.4min
[CV] END ....................model__C=1, model__gamma=0.0001; total time= 1.4min
[CV] END .....................model__C=1, model__gamma=0.001; total time= 4.1min
[CV] END .....................model__C=1, model__gamma=0.001; total time= 4.1min
[CV] END .....................model__C=1, model__gamma=0.001; total time= 4.1min
[CV] END .....................model__C=1, model__gamma=0.001; total time= 4.1min
[CV] END ......................model__C=1, model__gamma=0.01; total time= 4.9min
[CV] END ......................model__C=1, model__gamma=0.01; total time= 4.9min
[CV] END ......................model__C=1, model__gamma=0.01; total time= 5.0min
[CV] END ......................model__C=1, model

calculate the best result frme the "results" array.

dump the best model